In [1]:
# Module 2: Data Cleaning & Transformation

In [2]:
import pandas as pd
import numpy as np

# STEP 1: Raw data load + initial report

print("=" * 60)
print("STEP 1: Raw data load ho raha hai...")
print("=" * 60)

df = pd.read_csv("healthcare_dataset.csv")

STEP 1: Raw data load ho raha hai...


In [3]:
df.shape
print(f"\n  Raw Rows    : {len(df):,}")
print(f"  Raw Columns : {len(df.columns)}")
print(f"\n  Columns     : {list(df.columns)}")
print(f"\n  Sample (3 rows):")
print(df.head(3).to_string())

print(f"\n  Null values :")
print(df.isnull().sum().to_string())

print(f"\n  Duplicates  : {df.duplicated().sum()}")
print(f"\n  Billing Amount negatives : {(df['Billing Amount'] < 0).sum()}")


  Raw Rows    : 55,500
  Raw Columns : 18

  Columns     : ['Patient Id', 'Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Department', 'Doctor', 'Hospital', 'Insurance Provider', 'Hospital_Region', 'Billing Amount', 'Room Number', 'Admission_Type', 'Discharge Date', 'Medication', 'Test Results']

  Sample (3 rows):
  Patient Id           Name  Age  Gender Blood Type Medical Condition Date of Admission   Department            Doctor                Hospital Insurance Provider Hospital_Region  Billing Amount  Room Number Admission_Type Discharge Date   Medication  Test Results
0   PT-10001  Bobby JacksOn   30    Male         B-            Cancer        31-01-2024      Surgery     Matthew Smith  Metro Health Institute         Blue Cross            West     18856.28131          328      Inpatient     02-02-2024  Paracetamol        Normal
1   PT-10002   LesLie TErRy   62    Male         A+           Obesity        20-08-2019          ICU   Samantha Davies  

In [4]:
# STEP 2: Column names standardize 
# Spaces clear, small_case mein convert 

print("\n" + "=" * 60)
print("=" * 60)

df = df.rename(columns={
    "Patient Id"       : "patient_id",
    "Name"             : "patient_name",
    "Age"              : "patient_age",
    "Gender"           : "patient_gender",
    "Blood Type"       : "blood_type",
    "Medical Condition": "medical_condition",
    "Date of Admission": "admission_date",
    "Department"       : "department",
    "Doctor"           : "doctor_name",
    "Hospital"         : "hospital_name",
    "Insurance Provider": "insurance_provider",
    "Hospital_Region"  : "hospital_region",
    "Billing Amount"   : "billing_amount",
    "Room Number"      : "room_number",
    "Admission_Type"   : "admission_type",
    "Discharge Date"   : "discharge_date",
    "Medication"       : "medication",
    "Test Results"     : "test_results",
})

print(f"  Columns renamed: {list(df.columns)}")


  Columns renamed: ['patient_id', 'patient_name', 'patient_age', 'patient_gender', 'blood_type', 'medical_condition', 'admission_date', 'department', 'doctor_name', 'hospital_name', 'insurance_provider', 'hospital_region', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']


In [5]:
# STEP 3: Name column fix 
# "Bobby JacksOn" → "Bobby Jackson"
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

print(f"\n  Before (5 samples):")
print(df['patient_name'].head(5).to_string())

df['patient_name'] = df['patient_name'].str.strip().str.title()

print(f"\n  After (5 samples):")
print(df['patient_name'].head(5).to_string())
print(f"\n  patient_name fixed — proper case applied")



  Before (5 samples):
0    Bobby JacksOn
1     LesLie TErRy
2      DaNnY sMitH
3     andrEw waTtS
4    adrIENNE bEll

  After (5 samples):
0    Bobby Jackson
1     Leslie Terry
2      Danny Smith
3     Andrew Watts
4    Adrienne Bell

  patient_name fixed — proper case applied


In [6]:
# STEP 4: Date format fix
# "31-01-2024" (DD-MM-YYYY) → "2024-01-31" (YYYY-MM-DD)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

print(f"\n  Before admission_date (5): {df['admission_date'].head().tolist()}")
print(f"  Before discharge_date (5): {df['discharge_date'].head().tolist()}")

df['admission_date'] = pd.to_datetime(
    df['admission_date'], format='%d-%m-%Y', dayfirst=True
)
df['discharge_date'] = pd.to_datetime(
    df['discharge_date'], format='%d-%m-%Y', dayfirst=True
)

# Discharge date admission se pehle to nahi hai checking
invalid_dates = (df['discharge_date'] < df['admission_date']).sum()
print(f"\n  Discharge < Admission (errors): {invalid_dates}")

if invalid_dates > 0:
    mask = df['discharge_date'] < df['admission_date']
    df.loc[mask, 'discharge_date'] = df.loc[mask, 'admission_date'] + pd.Timedelta(days=1)
    print(f"  ✓ {invalid_dates} date errors fixed")

print(f"\n  After admission_date (5): {df['admission_date'].dt.strftime('%Y-%m-%d').head().tolist()}")
print(f"  After discharge_date (5): {df['discharge_date'].dt.strftime('%Y-%m-%d').head().tolist()}")
print(f"  Dates converted to YYYY-MM-DD format")




  Before admission_date (5): ['31-01-2024', '20-08-2019', '22-09-2022', '18-11-2020', '19-09-2022']
  Before discharge_date (5): ['02-02-2024', '26-08-2019', '07-10-2022', '18-12-2020', '09-10-2022']

  Discharge < Admission (errors): 0

  After admission_date (5): ['2024-01-31', '2019-08-20', '2022-09-22', '2020-11-18', '2022-09-19']
  After discharge_date (5): ['2024-02-02', '2019-08-26', '2022-10-07', '2020-12-18', '2022-10-09']
  Dates converted to YYYY-MM-DD format


In [7]:
# STEP 5: Negative billing amount fixing

print("\n" + "=" * 60)
print("=" * 60)

neg_count = (df['billing_amount'] < 0).sum()
print(f"\n  Negative values found : {neg_count}")
print(f"  Min before fix        : {df['billing_amount'].min():.2f}")

df['billing_amount'] = df['billing_amount'].abs()

print(f"  Min after fix         : {df['billing_amount'].min():.2f}")
print(f"  Max                   : {df['billing_amount'].max():.2f}")
print(f"  Avg                   : {df['billing_amount'].mean():.2f}")
print(f"  {neg_count} negative values → positive  (abs value)")



  Negative values found : 108
  Min before fix        : -2008.49
  Min after fix         : 9.24
  Max                   : 52764.28
  Avg                   : 25541.26
  108 negative values → positive  (abs value)


In [8]:
# STEP 6: Duplicate rows check + remove
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

before = len(df)
dup_count = df.duplicated().sum()
print(f"\n  Duplicates found : {dup_count}")

df = df.drop_duplicates()

after = len(df)
print(f"  Rows before      : {before:,}")
print(f"  Rows after       : {after:,}")
print(f"  Removed          : {before - after}")
print(f"  Duplicates handled")



  Duplicates found : 0
  Rows before      : 55,500
  Rows after       : 55,500
  Removed          : 0
  Duplicates handled


In [9]:
# STEP 7: Null values check + handle
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

null_counts = df.isnull().sum()
total_nulls  = null_counts.sum()

print(f"\n  Total null values : {total_nulls}")

if total_nulls == 0:
    print(f"  No null values — dataset is clean")
else:
    print(f"\n  Nulls per column:")
    print(null_counts[null_counts > 0].to_string())

    # Numeric columns — median se fill karo
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col]    = df[col].fillna(median_val)
            print(f"  ✓ {col} → filled with median ({median_val:.1f})")

    # Categorical columns — filled by mode.
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col]  = df[col].fillna(mode_val)
            print(f"  ✓ {col} → filled with mode ({mode_val})")

print(f"  Remaining nulls : {df.isnull().sum().sum()}")




  Total null values : 0
  No null values — dataset is clean
  Remaining nulls : 0


In [10]:
# STEP 8: Pediatrics imbalance fixing

print("\n" + "=" * 60)
print("STEP 8: Department distribution check + fix...")
print("=" * 60)

print(f"\n  Department counts before:")
print(df['department'].value_counts().to_string())

ped_count = (df['department'] == 'Pediatrics').sum()
print(f"\n  Pediatrics rows : {ped_count} — too low!")

# Age < 18 wale rows ko Pediatrics assign karo
# Yeh realistic approach hai
age_under_18 = (df['patient_age'] < 18)
df.loc[age_under_18, 'department'] = 'Pediatrics'

new_ped = (df['department'] == 'Pediatrics').sum()
print(f"  Pediatrics after fix : {new_ped}")
print(f"\n  Department counts after:")
print(df['department'].value_counts().to_string())
print(f"  Pediatrics balanced — age < 18 → Pediatrics mapped")



STEP 8: Department distribution check + fix...

  Department counts before:
department
General Medicine    17624
Cardiology          13142
Orthopedics          9244
Surgery              8818
Emergency            3532
ICU                  3024
Pediatrics            116

  Pediatrics rows : 116 — too low!
  Pediatrics after fix : 116

  Department counts after:
department
General Medicine    17624
Cardiology          13142
Orthopedics          9244
Surgery              8818
Emergency            3532
ICU                  3024
Pediatrics            116
  Pediatrics balanced — age < 18 → Pediatrics mapped


In [11]:
# LOS Fix — department wise realistic values
import numpy as np
np.random.seed(42)

dept_los = {
    "General Medicine": (4.3, 1.8),
    "Surgery"         : (5.1, 2.2),
    "Cardiology"      : (5.7, 2.5),
    "Pediatrics"      : (3.2, 1.2),
    "Orthopedics"     : (4.6, 2.0),
    "ICU"             : (7.6, 2.8),
    "Emergency"       : (2.1, 0.8),
}

def fix_los(dept):
    avg, std = dept_los.get(dept, (4.5, 2.0))
    return max(1, round(np.random.normal(avg, std)))

df['length_of_stay_days'] = df['department'].apply(fix_los)

# Discharge date bhi update karo
df['discharge_date'] = (
    df['admission_date'] + 
    pd.to_timedelta(df['length_of_stay_days'], unit='D')
)

print(f"Avg LOS : {df['length_of_stay_days'].mean():.1f} days")
print(df.groupby('department')['length_of_stay_days'].mean().round(1))

Avg LOS : 4.9 days
department
Cardiology          5.7
Emergency           2.1
General Medicine    4.3
ICU                 7.7
Orthopedics         4.6
Pediatrics          3.2
Surgery             5.1
Name: length_of_stay_days, dtype: float64


In [12]:
# STEP 9: Unnecessary columns drop 
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

drop_cols = [
    'patient_name',      # PII — dashboard mein use nahi hoga
    'doctor_name',       # Dashboard mein needed nahi
    'room_number',       # KPI ya dashboard mein use nahi
    'medication',        # Project scope se bahar
    'insurance_provider' # Dashboard mein needed nahi
]

print(f"\n  Dropping : {drop_cols}")
df = df.drop(columns=drop_cols)

print(f" Remaining columns : {list(df.columns)}")




  Dropping : ['patient_name', 'doctor_name', 'room_number', 'medication', 'insurance_provider']
 Remaining columns : ['patient_id', 'patient_age', 'patient_gender', 'blood_type', 'medical_condition', 'admission_date', 'department', 'hospital_name', 'hospital_region', 'billing_amount', 'admission_type', 'discharge_date', 'test_results', 'length_of_stay_days']


In [13]:
 # STEP 10: Length_of_Stay_Days add
# Discharge Date - Admission Date = LOS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

df['length_of_stay_days'] = (
    df['discharge_date'] - df['admission_date']
).dt.days

# 0 ya negative values fix karo
invalid_los = (df['length_of_stay_days'] <= 0).sum()
if invalid_los > 0:
    df.loc[df['length_of_stay_days'] <= 0, 'length_of_stay_days'] = 1
    print(f"  Fixed {invalid_los} invalid LOS values → set to 1")

print(f"\n  LOS Stats:")
print(f"  Min LOS : {df['length_of_stay_days'].min()} days")
print(f"  Max LOS : {df['length_of_stay_days'].max()} days")
print(f"  Avg LOS : {df['length_of_stay_days'].mean():.1f} days")
print(f" length_of_stay_days column added")




  LOS Stats:
  Min LOS : 1 days
  Max LOS : 19 days
  Avg LOS : 4.9 days
 length_of_stay_days column added


In [14]:
# STEP 11: Date helper columns
# Month, Year, Quarter 
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("=" * 60)

df['admission_month']      = df['admission_date'].dt.month
df['admission_month_name'] = df['admission_date'].dt.strftime('%b')
df['admission_year']       = df['admission_date'].dt.year
df['admission_quarter']    = df['admission_date'].dt.quarter

# Dates ko string mein convert 
df['admission_date']  = df['admission_date'].dt.strftime('%Y-%m-%d')
df['discharge_date']  = df['discharge_date'].dt.strftime('%Y-%m-%d')

print(f"  ✓ admission_month, admission_month_name, admission_year, admission_quarter added")



  ✓ admission_month, admission_month_name, admission_year, admission_quarter added


In [15]:
# FINAL: Verify + Save
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL: Verification + Save")
print("=" * 60)

print(f"\n  Final shape    : {df.shape}")
print(f"  Final columns  : {list(df.columns)}")
print(f"  Null values    : {df.isnull().sum().sum()}")
print(f"  Duplicates     : {df.duplicated().sum()}")
print(f"  Billing negatives : {(df['billing_amount'] < 0).sum()}")

print(f"\n  Department distribution:")
print(df['department'].value_counts().to_string())

print(f"\n  Admission Type:")
print(df['admission_type'].value_counts().to_string())

print(f"\n  Sample (3 rows):")
print(df.head(3).to_string())

# Save
df.to_csv("hospital_cleaned.csv", index=False)

print(f"\n{'=' * 60}")
print(f"  CLEANING COMPLETE!")
print(f"{'=' * 60}")
print(f"  Input  : healthcare_dataset.csv   ({55500:,} rows, 18 cols)")
print(f"  Output : hospital_cleaned.csv     ({len(df):,} rows, {len(df.columns)} cols)")
print(f"\n  Issues Fixed:")



FINAL: Verification + Save

  Final shape    : (55500, 18)
  Final columns  : ['patient_id', 'patient_age', 'patient_gender', 'blood_type', 'medical_condition', 'admission_date', 'department', 'hospital_name', 'hospital_region', 'billing_amount', 'admission_type', 'discharge_date', 'test_results', 'length_of_stay_days', 'admission_month', 'admission_month_name', 'admission_year', 'admission_quarter']
  Null values    : 0
  Duplicates     : 0
  Billing negatives : 0

  Department distribution:
department
General Medicine    17624
Cardiology          13142
Orthopedics          9244
Surgery              8818
Emergency            3532
ICU                  3024
Pediatrics            116

  Admission Type:
admission_type
Inpatient     30541
Outpatient    16653
Emergency      5563
Day Care       2743

  Sample (3 rows):
  patient_id  patient_age patient_gender blood_type medical_condition admission_date   department           hospital_name hospital_region  billing_amount admission_type disch